# 00 · Smoke Test — Data Splits

Quick sanity check that the **loan-stratified temporal split** is usable end-to-end, *before*
investing in the foundation model. Trains an XGBoost baseline to confirm:

1. `data/processed/{train,val,test}.parquet` load and align with `tokenizer.yaml`.
2. A forward-looking default label can be built.
3. There is real predictive signal (AUC >> 0.5), and **val ~= test** (the temporal split generalizes).

**Prerequisite:** run the split first — `python scripts/prepare_data.py` — so the parquets exist.

> Smoke test only, not the Gate-G1 baseline. The real baseline (Phase B) conditions on
> *performing-at-observation* and reports the full metric set. See the caveats at the bottom.

In [1]:
import yaml
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

# features come straight from the generated tokenizer config
cfg = yaml.safe_load(open('../configs/dutch_mortgages/tokenizer.yaml'))
NUM = cfg['profile'].get('numeric', []) + cfg['event'].get('numeric', [])
CAT = (cfg['profile'].get('categorical', []) + cfg['profile'].get('flags', [])
       + cfg['event'].get('categorical', []) + cfg['event'].get('flags', []))
FEATS = NUM + CAT
print(f'{len(FEATS)} features ({len(NUM)} numeric, {len(CAT)} categorical)')

42 features (18 numeric, 24 categorical)


## Label: CRR default within the next 6 months

Observe each loan at **month 12** (`2024-12-31`); label = 1 if `default_crr_flag` becomes `Y`
in any of the next 6 monthly cutoffs (`2025-01`…`2025-06`). One row per loan -> matches the
loan-level split, no within-loan correlation.

In [2]:
OBS = '2024-12-31'
FWD = ['2025-01-31', '2025-02-28', '2025-03-31', '2025-04-30', '2025-05-31', '2025-06-30']
COLS = list(set(FEATS + ['loan_id', 'reporting_date', 'default_crr_flag']))

def build(path):
    df = pd.read_parquet(path, columns=COLS)
    obs = df[df.reporting_date == OBS].copy()
    fut = df[df.reporting_date.isin(FWD)]
    defaulted = set(fut.loc[fut.default_crr_flag == 'Y', 'loan_id'])
    y = obs.loan_id.isin(defaulted).astype(int).values
    return obs[FEATS].copy(), y

Xtr, ytr = build('../data/processed/train.parquet')
Xva, yva = build('../data/processed/val.parquet')
Xte, yte = build('../data/processed/test.parquet')
print(f'rows  train {len(ytr):,}  val {len(yva):,}  test {len(yte):,}')
print(f'default rate  train {ytr.mean():.3%}  val {yva.mean():.3%}  test {yte.mean():.3%}')

rows  train 374,764  val 46,897  test 46,875
default rate  train 2.348%  val 2.282%  test 2.357%


## Encode + train

Categoricals are mapped to integer codes fit **on train only** (unseen values -> NaN), so no
val/test information leaks into the encoding.

In [3]:
cats_map: dict = {}

def encode(X, fit=False):
    X = X.copy()
    for c in CAT:
        if fit:
            cats_map[c] = {v: i for i, v in enumerate(pd.Series(X[c].dropna().unique()))}
        X[c] = X[c].map(cats_map[c]).astype('float')
    for c in NUM:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    return X

Xtr, Xva, Xte = encode(Xtr, fit=True), encode(Xva), encode(Xte)

model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
                          n_jobs=-1, tree_method='hist')
model.fit(Xtr, ytr)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, random_state=None, ...)

In [4]:
for name, X, y in [('val', Xva, yva), ('test', Xte, yte)]:
    p = model.predict_proba(X)[:, 1]
    print(f'{name:>4}:  ROC-AUC {roc_auc_score(y, p):.4f}   PR-AUC {average_precision_score(y, p):.4f}')

 val:  ROC-AUC 0.8896   PR-AUC 0.7807
test:  ROC-AUC 0.8949   PR-AUC 0.7792


## Expected result & caveats

Reference run: **ROC-AUC ~= 0.89, PR-AUC ~= 0.78** on both val and test (default rate ~= 2.3%).
val ~= test confirms the temporal split generalizes.

**This is a smoke test, not Gate G1:**

- *Synthetic data is easy* — the generator is rule-based, so XGBoost scores high. Real credit
  baselines are typically ~0.70–0.80; the foundation-model lift here may look modest.
- *Contemporaneous state inflates it* — month-12 `arrears_bucket`/`performing_status`/
  `default_crr_flag` are features, and some loans are already in default at month 12. The real
  baseline should **condition on performing-at-observation** (predict *new* defaults only).

The proper baseline (forward-window label + performing filter + full metric set) lands in
`scripts/train_baseline.py` / `src/credit_fm/data/label_generators.py` in Phase B.